# Duolingo Hungarian Vocabulary → Anki Deck

Scrapes vocabulary from [duome.eu](https://duome.eu/vocabulary/en/hu) and writes an Anki `.apkg` file using [genanki](https://github.com/kerrickstaley/genanki).

In [ ]:
# %pip install requests beautifulsoup4 genanki

In [ ]:
import requests
from bs4 import BeautifulSoup

URL = "https://duome.eu/vocabulary/en/hu"
resp = requests.get(URL)
resp.raise_for_status()
soup = BeautifulSoup(resp.text, "html.parser")

# The page uses List.js with valueNames: wN, wA, wP, wG, wT
# wN = word/phrase in Hungarian, wT = English translation
# Inspect what classes are actually present
sample = soup.select(".wN")[:3]
for s in sample:
    print(repr(s))
    print(s.parent.prettify()[:500])
    print("---")

In [ ]:
rows = []
for item in soup.select(".wN"):
    parent = item.parent
    hungarian = item.get_text(strip=True)
    trans_el = parent.select_one(".wT")
    translation = trans_el.get_text(strip=True) if trans_el else ""
    if hungarian and translation:
        rows.append((hungarian, translation))

print(f"Parsed {len(rows)} vocabulary entries")
rows[:10]

In [ ]:
import genanki
import random

MODEL_ID = 1607392319  # arbitrary stable id
DECK_ID = 2059400110   # arbitrary stable id

model = genanki.Model(
    MODEL_ID,
    "Duolingo HU-EN",
    fields=[
        {"name": "Hungarian"},
        {"name": "English"},
    ],
    templates=[
        {
            "name": "HU → EN",
            "qfmt": "{{Hungarian}}",
            "afmt": '{{FrontSide}}<hr id="answer">{{English}}',
        },
        {
            "name": "EN → HU",
            "qfmt": "{{English}}",
            "afmt": '{{FrontSide}}<hr id="answer">{{Hungarian}}',
        },
    ],
)

deck = genanki.Deck(DECK_ID, "Duolingo Hungarian")

for hu, en in rows:
    note = genanki.Note(model=model, fields=[hu, en])
    deck.add_note(note)

OUTPUT = "duolingo_hungarian.apkg"
genanki.Package(deck).write_to_file(OUTPUT)
print(f"Wrote {len(rows)} notes to {OUTPUT}")